# 004_meeting_sb_lens — Sales Brief 鏡頭表＋面向映射種子＋View（DDL）

- 用途：W2 Sales Brief 鏡頭（規格 §3.4 / §3.5 / §7.1 / §7.2）的一次性 migration。建標註表 `g_meeting_items_sb_bu`、對照格表 `g_meeting_sb_reconcile_bu`、面向映射種子表 `s_meeting_sb_facet_map`（從 [c05] 內嵌 CSV MERGE）與四個 View。資料由之後的 W2 T3 / T4 job 寫入；本支只建結構與種子。
- **正本是 [c04] `table_specs`、[c05] `seed_data`、[c06] `view_specs`**。改表欄 = 開新流水號 migration 做 `ALTER TABLE ADD COLUMNS`；改種子 / View 改 [c05] / [c06] 後重跑即可（先 `dry_run = true` 看差異）。
- 標註表**一列一個 `(item_key, app_name)`**（規格 §3.4 寫 `item_id`；比照 003：中立層重萃後 `item_id` 會變、`item_key` 不變，T3 整月重標時列不漂移）。對照格表鍵 `(app_name, brand, facet, brief_month)`；PM 審核欄 `reviewer / review_verdict / review_relation / review_note` 由 T4 MERGE **不覆蓋**。
- `brief_month` 窗口（STATUS D6）：不是日曆月，T3 以 `v_meeting_sb_period` 的 `submitted_end`（每 app × 期別 `max(submitted_at)`）為截止點推 `window_start / window_end`，兩張表都記下來。
- Sales Brief 文字端（待決 #10 已解）：品牌格 `g_sales_brief_brand_summary`、ALL 格 `g_sales_brief_app_summary`；四面向 ↔ `summary_futureOutlook / techProduct / pricing / otherInfo`。`v_meeting_sb_brief_text` 把兩張表轉成一格一列。
- Phase 1 不做 L2 fallback（STATUS §6.4 第 6 點）：`facet_basis` 只會是 `rule_primary / rule_secondary / unresolved`，`llm_fallback` 保留給之後。
- 冪等：表 `CREATE TABLE IF NOT EXISTS`；種子 MERGE（拿掉的列標 `is_active = false`）；View `CREATE OR REPLACE VIEW`。前置：002 的 `g_meeting_items_bu`、Sales Brief 三張表（`g_sales_brief_brand_summary` / `app_summary` / `market_intelligence`）要存在。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證 spec 與種子、印 DDL 與種子差異，不建任何物件）
- 負責人 / 更新日期：（填）/ 2026-10-23

| 物件 | 鍵 / 來源 | 內容 | 敏感度 |
|---|---|---|---|
| `g_meeting_items_sb_bu` | `(item_key, app_name)` | §3.4：brief_month（＋窗口）、app_name、brand / brand_in_sb、facet / facet_basis、sb_relevant / relevance_basis ＋ 中立層複製欄 ＋ lineage | S0–S2 |
| `g_meeting_sb_reconcile_bu` | `(app_name, brand, facet, brief_month)` | §3.5：sb_text、meeting_summary（含 S2，PM 內部）、meeting_summary_shareable（可外送）、relation、跨客戶訊號、PM 審核欄、lineage | S0–S2 |
| `s_meeting_sb_facet_map` | `topic_tag` | §7.1 四面向映射 ＋ §7.2 相關性規則代號 ＋ `schedule → product_plan` 跟隨；正本 [c05] | S0 |
| `v_meeting_sb_brief_text` | brand_summary ∪ app_summary | 一格一列：app_name × brand（`ALL` = app 層）× facet × brief_month → sb_text / sb_source | 依 Sales Brief 表 |
| `v_meeting_sb_period` | brand_summary ⋈ market_intelligence ⋈ app_summary | 每 app × brief_month：品牌清單、`submitted_start / submitted_end`、上一期截止（D6 窗口的材料） | S0 |
| `v_meeting_items_sb_q` | 標註表 ⋈ items | 標註列（active）＋ quote / numbers / model_nos / time_refs；T4 的會議端輸入、PM 看證據 | S0–S2 |
| `v_meeting_sb_reconcile_q` | 對照格表 | 加 `relation_rank`（§4.6 例外清單順序）、`effective_relation`、`review_pending`、`is_app_level` | S0–S2 |


In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run})


In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c06] 不碰 spark / dbutils，可被 tests/ 載入。
import csv
import io
import json
import re

SLUG = "004_meeting_sb_lens"
UNIT = "bu"  # 單位代碼：表名後綴（規格 §3 命名規則）
COL_RE = r"^[a-z][a-z0-9_]*$"
TYPE_RE = r"^(STRING|BIGINT|INT|DOUBLE|BOOLEAN|DATE|TIMESTAMP|ARRAY<STRING>)$"

# Sales Brief 四面向（規格 §7.1）↔ 填報表欄名（Phase 0 探勘，待決 #10 已解）。
# brand_summary / app_summary 兩張表都有這四欄；market_intelligence 的 field_key 是去掉 summary_ 前綴的同四值。
FACETS = ("market_outlook", "product_tech", "price", "other")
SB_FACET_COLS = {
    "market_outlook": "summary_futureOutlook",
    "product_tech": "summary_techProduct",
    "price": "summary_pricing",
    "other": "summary_otherInfo",
}
SB_FIELD_KEYS = {f: c.removeprefix("summary_") for f, c in SB_FACET_COLS.items()}  # market_intelligence.field_key
APPS = ("TV", "NB", "MNT")
APP_UNRESOLVED = "app_unresolved"  # 中立層 app = ALL / NULL 且客戶預設推不出唯一 app（§4.3 第 2 點）；不進對照，只計數
BRAND_ALL = "ALL"  # 對照格表 brand = ALL 的列是 app 層格（§3.5），對應 g_sales_brief_app_summary
RELATIONS = ("corroborate", "supplement", "conflict", "meeting_only", "sb_only")
RELATION_RANK = {"conflict": 1, "supplement": 2, "meeting_only": 3, "corroborate": 4, "sb_only": 5}  # §4.6 例外清單順序
REVIEW_VERDICTS = ("agree", "fix_relation", "drop")
FACET_BASIS = ("rule_primary", "rule_secondary", "llm_fallback", "unresolved")  # Phase 1 不會出現 llm_fallback
# §7.2 相關性規則代號（種子 relevance_rule 欄的允許值；T3 程式照代號判，不另寫 if）：
#   always                      = 規則 1：含該 tag 即相關
#   spec_statement              = 規則 2：spec 且 statement_type ∈ {plan, decision, request}
#   capacity_timeref            = 規則 3：capacity 且 time_refs 非空
#   quality_request_or_customer = 規則 4：quality 且（statement_type = request 或 speaker_side = customer）
#   never                       = 不映射的 tag（schedule / engineering / terminology / admin）；單獨出現一律 false
RELEVANCE_RULES = ("always", "spec_statement", "capacity_timeref", "quality_request_or_customer", "never")
FACET_MAP_VERSION = "0.1"  # 改 [c05] 種子就升；T3 把它寫進 labeler_version
FACET_MAP_COLS = ["topic_tag", "facet", "relevance_rule", "follow_tag", "note"]

# T3 / T4 依賴的欄一定要有
REQUIRED_COLS = {
    "g_meeting_items_sb_bu": [
        "item_key", "item_id", "brief_month", "window_start", "window_end", "app_name", "brand", "brand_in_sb",
        "facet", "facet_basis", "sb_relevant", "relevance_basis", "active", "labeler_version",
    ],
    "g_meeting_sb_reconcile_bu": [
        "app_name", "brand", "facet", "brief_month", "window_start", "window_end", "sb_text", "sb_source",
        "meeting_summary", "meeting_summary_shareable", "shareable_failed", "meeting_item_ids", "relation",
        "relation_basis", "reviewer", "review_verdict", "review_relation", "review_note",
    ],
    "s_meeting_sb_facet_map": ["topic_tag", "facet", "relevance_rule", "follow_tag", "is_active"],
}
# S3 欄位：只准落在 s_meeting_record_mail_bu / s_meeting_participants_bu；任何 View 不得暴露。
S3_COLS = {
    "display_name_raw", "name_zh", "name_en", "email", "emp_id", "account",
    "forwarder_name_zh", "forwarder_emp_id", "forwarder_account",
    "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display",
}


In [ ]:
# [c03] pure_ddl
# spec → SQL 字串的純函式；不碰 spark。測試在本機用同一份 spec 建空表跑 View 的 SELECT。
def q(s: str | None) -> str:
    """SQL 單引號字串常值；None → ''。"""
    return "'" + (s or "").replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_create_table(fq: str, spec: dict) -> str:
    """CREATE TABLE IF NOT EXISTS；鍵欄 NOT NULL；有 cluster_by 才加 CLUSTER BY（liquid clustering）。"""
    keys = set(spec["key"])
    cols = ",\n".join(
        f"    {name} {typ}{' NOT NULL' if name in keys else ''} COMMENT {q(comment)}"
        for name, typ, comment in spec["columns"]
    )
    cluster = f"\nCLUSTER BY ({', '.join(spec['cluster_by'])})" if spec.get("cluster_by") else ""
    return f"CREATE TABLE IF NOT EXISTS {fq} (\n{cols}\n) USING DELTA{cluster}\nCOMMENT {q(spec['comment'])}"


def build_select(spec: dict, fq) -> str:
    """View 的查詢本體：spec['select'] = [(expr, alias, comment)]，spec['body'](fq) 回傳 FROM 起的 SQL。
    fq(table_name) -> 可查詢的名稱；正式是三段式，測試用 temp view 名。"""
    select = ",\n".join(f"    {expr} AS {alias}" for expr, alias, _ in spec["select"])
    return f"SELECT\n{select}\n{spec['body'](fq)}"


def build_create_view(fq_view: str, spec: dict, fq) -> str:
    """CREATE OR REPLACE VIEW，欄位清單帶 COMMENT（Genie / 人看的語意）。"""
    col_list = ",\n".join(f"    {alias} COMMENT {q(comment)}" for _, alias, comment in spec["select"])
    return (
        f"CREATE OR REPLACE VIEW {fq_view} (\n{col_list}\n)\nCOMMENT {q(spec['comment'])}\nAS\n"
        + build_select(spec, fq)
    )


def schema_ddl(spec: dict) -> str:
    """欄位 → 'a BIGINT, b STRING'（測試建空表、[c30] 比對型別用）。"""
    return ", ".join(f"{n} {t}" for n, t, _ in spec["columns"])


def add_columns_sql(fq: str, spec: dict, missing: list[str]) -> str:
    """[c30] 發現缺欄時印出的補欄語句（不自動執行；要補就開新流水號 migration）。"""
    by_name = {n: (t, c) for n, t, c in spec["columns"]}
    parts = ", ".join(f"{n} {by_name[n][0]} COMMENT {q(by_name[n][1])}" for n in missing)
    return f"ALTER TABLE {fq} ADD COLUMNS ({parts})"


def validate_specs(tables: dict, views: dict) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。"""
    errs = []
    for t, spec in tables.items():
        names = [c[0] for c in spec["columns"]]
        dup = sorted({n for n in names if names.count(n) > 1})
        if dup:
            errs.append(f"{t} 欄位重複：{dup}")
        for n, typ, comment in spec["columns"]:
            if not re.match(COL_RE, n):
                errs.append(f"{t}.{n} 欄名格式不符（小寫英數底線）")
            if not re.match(TYPE_RE, typ):
                errs.append(f"{t}.{n} 型別不在允許清單：{typ}")
            if not comment:
                errs.append(f"{t}.{n} 沒有 COMMENT")
        for k in list(spec["key"]) + list(spec.get("cluster_by", [])):
            if k not in names:
                errs.append(f"{t} key / cluster_by 欄 {k} 不存在")
        for r in REQUIRED_COLS.get(t, []):
            if r not in names:
                errs.append(f"{t} 缺必要欄 {r}")
        if not spec.get("comment"):
            errs.append(f"{t} 沒有表 COMMENT")
    for v, spec in views.items():
        aliases = [a for _, a, _ in spec["select"]]
        dup = sorted({a for a in aliases if aliases.count(a) > 1})
        if dup:
            errs.append(f"{v} 輸出欄重複：{dup}")
        leak = sorted(S3_COLS & set(aliases))
        if leak:
            errs.append(f"{v} 暴露 S3 欄：{leak}")
        for _, a, c in spec["select"]:
            if not re.match(COL_RE, a):
                errs.append(f"{v}.{a} 欄名格式不符")
            if not c:
                errs.append(f"{v}.{a} 沒有 COMMENT")
        if not spec.get("comment"):
            errs.append(f"{v} 沒有 View COMMENT")
    return errs


# ---------- 種子（面向映射表）----------
def parse_csv_text(text: str, expect_cols: list[str]) -> list[dict]:
    """CSV 格式文字 → list[dict]；欄位順序必須等於 expect_cols，值去頭尾空白，空字串 → None。（同 001 [c03]）"""
    reader = csv.DictReader(io.StringIO(text.lstrip("﻿").strip()))
    if reader.fieldnames != expect_cols:
        raise ValueError(f"欄位應為 {expect_cols}，實際 {reader.fieldnames}")
    return [{k: (v.strip() or None) if isinstance(v, str) else v for k, v in r.items()} for r in reader]


def validate_facet_seed(rows: list[dict], expect_tags: set[str] | None = None) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。行號 = [c05] 該段第幾行（1 是欄名）。
    expect_tags 給 L1 詞表（task 3 [c02] TOPIC_TAGS）時，種子必須剛好涵蓋全部 tag（多一個少一個都報）。"""
    errs = []
    by_tag = {}
    for i, r in enumerate(rows, start=2):
        tag = r["topic_tag"] or ""
        if not re.match(COL_RE, tag):
            errs.append(f"FACET_MAP_SEED 第 {i} 行 topic_tag 格式不符（小寫英數底線）：{tag!r}")
        if tag in by_tag:
            errs.append(f"FACET_MAP_SEED 第 {i} 行 topic_tag 重複：{tag}")
        by_tag[tag] = r
        if r["facet"] is not None and r["facet"] not in FACETS:
            errs.append(f"FACET_MAP_SEED 第 {i} 行 facet 不在 {list(FACETS)}（或留空 = 不映射）：{r['facet']!r}")
        if r["relevance_rule"] not in RELEVANCE_RULES:
            errs.append(f"FACET_MAP_SEED 第 {i} 行 relevance_rule 不在 {list(RELEVANCE_RULES)}：{r['relevance_rule']!r}")
        if r["facet"] is None and r["relevance_rule"] != "never":
            errs.append(f"FACET_MAP_SEED 第 {i} 行 不映射的 tag 只能是 never：{tag}")
        if r["facet"] is not None and r["relevance_rule"] == "never":
            errs.append(f"FACET_MAP_SEED 第 {i} 行 有映射的 tag 不能是 never（§7.2 每個映射 tag 都有規則）：{tag}")
        if r["follow_tag"] is not None and r["facet"] is not None:
            errs.append(f"FACET_MAP_SEED 第 {i} 行 follow_tag 只給不映射的 tag 用：{tag}")
    for tag, r in by_tag.items():
        ft = r["follow_tag"]
        if ft is not None and (ft not in by_tag or by_tag[ft]["facet"] is None):
            errs.append(f"FACET_MAP_SEED topic_tag {tag} 的 follow_tag {ft!r} 不是有映射的 tag")
    if expect_tags is not None:
        missing = sorted(set(expect_tags) - set(by_tag))
        extra = sorted(set(by_tag) - set(expect_tags))
        if missing:
            errs.append(f"FACET_MAP_SEED 少了 L1 詞表的 tag：{missing}")
        if extra:
            errs.append(f"FACET_MAP_SEED 有 L1 詞表沒有的 tag：{extra}")
    return errs


def facet_lookup(rows: list[dict]) -> dict[str, dict]:
    """topic_tag → {facet, relevance_rule, follow_tag}；T3 的純函式 cell 用同一份種子（從表讀回或直接用常數）。"""
    return {r["topic_tag"]: {k: r[k] for k in ("facet", "relevance_rule", "follow_tag")} for r in rows}


In [ ]:
# [c04] table_specs
# 正本。欄位 = (名稱, 型別, COMMENT)。鍵欄自動 NOT NULL。
# g_meeting_items_sb_bu   = 規格 §3.4（鍵改 (item_key, app_name)，比照 003：重萃後 item_id 變、item_key 不變）＋ D6 窗口欄 ＋ 中立層複製欄。
# g_meeting_sb_reconcile_bu = 規格 §3.5 ＋ D6 窗口欄 ＋ 筆數欄；PM 審核欄 T4 MERGE 不覆蓋。
# s_meeting_sb_facet_map  = 規格 §7.1 + §7.2（config/sb_facet_map.yaml 的 Delta 版，D2 不用 yaml）；正本 [c05]。
# CLUSTER BY (brief_month)：T3 / T4 按月整批重標 / 重對照，PM 按月審。
TABLES = {
    "g_meeting_items_sb_bu": {
        "comment": "會議條目的 Sales Brief 鏡頭（W2 T3 產出）：一列一個 (item_key, app_name)，條目歸入哪一期 brief、對到哪個 Sales Brief 品牌、哪個面向、是否進對照。整月重標覆寫；不含姓名 email。正本：004_meeting_sb_lens [c04]。",
        "key": ["item_key", "app_name"],
        "cluster_by": ["brief_month"],
        "columns": [
            ("item_key", "STRING", "承接鍵（= g_meeting_items_bu.item_key）；重萃不變"),
            ("app_name", "STRING", "TV / NB / MNT / app_unresolved；中立層 app = ALL 或 NULL 時依客戶 app_default 展開成多列，推不出唯一值標 app_unresolved（不進對照）"),
            ("item_id", "STRING", "目前對應的中立條目 id；重萃後由 T3 更新"),
            ("pk_seq", "BIGINT", "回溯：來源信"),
            ("segment_no", "INT", "回溯：段序"),
            ("sub_no", "INT", "回溯：子段序"),
            ("unit", "STRING", "單位代碼（本版 bu）"),
            ("brief_month", "STRING", "歸入哪一期 Sales Brief，yyyy-MM；由 event_date 落在哪個窗口決定（D6），不是日曆月"),
            ("brief_month_basis", "STRING", "窗口怎麼算的：submitted_at（v_meeting_sb_period 的截止點）/ calendar / fixed_day / override"),
            ("window_start", "DATE", "該期窗口起（含）：上一期截止日 + 1"),
            ("window_end", "DATE", "該期窗口迄（含）：截止日 + grace_days"),
            ("app_basis_sb", "STRING", "app_name 來源：neutral（中立層 app 有值）/ customer_default（ALL / NULL 依 s_semantic_entity.app_default 展開）/ unresolved"),
            ("customer", "STRING", "正規化客戶 entity_id（複製自中立層）；samsung 且 app ∈ {TV, MNT} 視同 samsung_vd（D9）"),
            ("brand", "STRING", "Sales Brief 品牌名（s_semantic_entity_alias 的 sb_brand，例 Samsung / Asus）；對不上時保留 customer 原值，brand_in_sb = false；customer NULL 時 NULL"),
            ("brand_basis", "STRING", "brand 來源：sb_brand（別名表）/ samsung_vd_rule（D9）/ customer_fallback（無 sb_brand，保留 entity_id）/ none（customer NULL）"),
            ("brand_in_sb", "BOOLEAN", "該品牌是否出現在該期該 app 的 Sales Brief（v_meeting_sb_period.brands）；false 的條目仍可進 meeting_only 格"),
            ("facet", "STRING", "market_outlook / product_tech / price / other；NULL = 推不出（主標籤與其餘標籤都不映射）"),
            ("facet_basis", "STRING", "rule_primary（topic_tag_primary 直接命中）/ rule_secondary（主標籤不映射、其餘 tag 或 follow_tag 命中）/ llm_fallback（Phase 1 不用）/ unresolved"),
            ("sb_relevant", "BOOLEAN", "是否進對照（§7.2 四條規則，程式判）；app_unresolved 或 facet NULL 時一律 false"),
            ("relevance_basis", "STRING", "命中哪條：always / spec_statement / capacity_timeref / quality_request_or_customer，加命中的 tag（例 always:demand）；不相關 → never 或 no_rule"),
            ("statement_type", "STRING", "複製自中立層（§7.2 規則 2 / 4 用、審核用）"),
            ("topic_tag_primary", "STRING", "複製自中立層"),
            ("topic_tags", "ARRAY<STRING>", "複製自中立層；對照格 topic_tags_agg 由此聚合"),
            ("speaker_side", "STRING", "複製自中立層（§7.2 規則 4 用）"),
            ("event_date", "DATE", "複製自中立層；窗口歸期依此"),
            ("summary", "STRING", "一句摘要（複製自中立層），PM 看清單不必回 join"),
            ("sensitivity", "STRING", "S0 / S1 / S2（複製自中立層）；對照格 sensitivity_max 與 shareable 驗證都要看"),
            ("confidence", "DOUBLE", "L1 自評（複製自中立層）；T4 每格最多帶 8 條依此降冪"),
            ("active", "BOOLEAN", "中立層目前仍有此 item_key 的有效條目且仍落在本期；整月重標時不在名單的舊列標 false，不刪"),
            ("labeler_version", "STRING", "lineage：T3 規則版本 + 映射表版本（例 0.1+map0.1）；映射改版 → 整月重標"),
            ("labeled_at", "TIMESTAMP", "T3 標註時間"),
            ("updated_at", "TIMESTAMP", "最近一次 MERGE 時間"),
        ],
    },
    "g_meeting_sb_reconcile_bu": {
        "comment": "Sales Brief 對照格（W2 T4 產出）：一列一格 app × brand × facet × brief_month；brand = ALL 是 app 層格（對應 g_sales_brief_app_summary）。meeting_summary 含 S2 只給 PM 內部，meeting_summary_shareable 才可外送。PM 審核欄 T4 不覆蓋。正本：004_meeting_sb_lens [c04]。",
        "key": ["app_name", "brand", "facet", "brief_month"],
        "cluster_by": ["brief_month"],
        "columns": [
            ("app_name", "STRING", "TV / NB / MNT"),
            ("brand", "STRING", "Sales Brief 品牌名（同 g_meeting_items_sb_bu.brand）；ALL = app 層格"),
            ("facet", "STRING", "market_outlook / product_tech / price / other"),
            ("brief_month", "STRING", "yyyy-MM"),
            ("window_start", "DATE", "會議端取條目的窗口起（D6；與標註表一致）"),
            ("window_end", "DATE", "會議端取條目的窗口迄"),
            ("sb_text", "STRING", "Sales Brief 該格原文：品牌格取 g_sales_brief_brand_summary，ALL 格取 g_sales_brief_app_summary；NULL = 無填寫或該期無此品牌"),
            ("sb_source", "STRING", "文字來源表名（g_sales_brief_brand_summary / g_sales_brief_app_summary）或 none"),
            ("meeting_summary", "STRING", "L3 產出的會議端 1–3 句綜合（含 S2，PM 內部用）；只有會議的格子用摘要模式批次產"),
            ("meeting_summary_shareable", "STRING", "可外送版：不含 S2 條目的數字 / 量價 / 限簽 / Allocation 細節；程式驗證失敗 → NULL 並 shareable_failed = true"),
            ("shareable_failed", "BOOLEAN", "shareable 驗證失敗（S2 條目 numbers.value 出現在 shareable 內）"),
            ("meeting_item_ids", "ARRAY<STRING>", "證據條目 item_id（送 LLM 的最多 8 條 ＋ 未送的都列）"),
            ("meeting_item_keys", "ARRAY<STRING>", "同上的 item_key（重萃後仍能對回）"),
            ("n_items", "INT", "格內 sb_relevant 條目數"),
            ("n_items_sent", "INT", "實際送 LLM 的條目數（≤ 8，依 confidence 降冪）；超過的只計數"),
            ("topic_tags_agg", "ARRAY<STRING>", "格內條目 topic_tag_primary 去重；畫面在格內分組用（other 格尤其需要）"),
            ("relation", "STRING", "corroborate / supplement / conflict / meeting_only / sb_only"),
            ("relation_basis", "STRING", "rule（單邊為空直接判）/ llm（兩邊都有，L3 判）"),
            ("rationale", "STRING", "一句理由（L3）"),
            ("number_caveat", "STRING", "兩邊數字可否比較的說明（單位 / 口徑不同時不得判 conflict）"),
            ("cross_customer_flag", "BOOLEAN", "品牌格：是否參與 app 層的共同訊號（≥ 2 品牌共享）"),
            ("cross_customer_note", "STRING", "ALL 格：跨客戶訊號本文（signals 串成）；品牌格 NULL"),
            ("sensitivity_max", "STRING", "格內條目最高敏感級 S0 / S1 / S2"),
            ("reviewer", "STRING", "PM 填。**T4 MERGE 不覆蓋**"),
            ("review_verdict", "STRING", "PM 填：agree / fix_relation / drop。**T4 MERGE 不覆蓋**"),
            ("review_relation", "STRING", "PM 填：fix_relation 時的正確 relation（few-shot 素材）。**T4 MERGE 不覆蓋**"),
            ("review_note", "STRING", "PM 填。**T4 MERGE 不覆蓋**"),
            ("review_carried_from", "STRING", "審核欄由哪個舊格承接（鍵相同時直接沿用，此欄 NULL；保留給鍵規則變動時）"),
            ("prompt_version", "STRING", "lineage：L3 prompt 版本；rule 判的格子也記（表示哪一版流程產的）"),
            ("model", "STRING", "lineage：模型名；純規則寫入為 rule"),
            ("created_at", "TIMESTAMP", "首次產生時間"),
            ("updated_at", "TIMESTAMP", "最近一次 MERGE 時間"),
        ],
    },
    "s_meeting_sb_facet_map": {
        "comment": "Sales Brief 四面向映射（§7.1）＋ 相關性規則代號（§7.2）：一列一個受控 topic_tag。facet NULL = 不映射；follow_tag = 與該 tag 共存時隨它映入（schedule → product_plan）。正本：004_meeting_sb_lens [c05] FACET_MAP_SEED。",
        "key": ["topic_tag"],
        "columns": [
            ("topic_tag", "STRING", "受控主題標籤（task 3 [c02] TOPIC_TAGS 的 14 個）"),
            ("facet", "STRING", "market_outlook / product_tech / price / other；NULL = 不映射"),
            ("relevance_rule", "STRING", "always / spec_statement / capacity_timeref / quality_request_or_customer / never（§7.2 規則代號，見 [c02]）"),
            ("follow_tag", "STRING", "不映射的 tag 若與此 tag 共存則隨它映入（§7.1：schedule 隨 product_plan）；其餘 NULL"),
            ("note", "STRING", "備註"),
            ("is_active", "BOOLEAN", "種子拿掉的列標 false，不刪"),
            ("updated_at", "TIMESTAMP", "最近一次 MERGE 時間"),
        ],
    },
    "b_meeting_migration_log": {
        "comment": "會議萃取 migration 執行紀錄（只增不改）",
        "key": ["slug", "run_at"],
        "columns": [
            ("slug", "STRING", "migration 流水號＋名稱（例 002_meeting_neutral_tables）"),
            ("run_at", "TIMESTAMP", "執行時間"),
            ("run_by", "STRING", "current_user()"),
            ("result", "STRING", "JSON：各物件 created / exists / replaced"),
            ("note", "STRING", "備註"),
        ],
    },
}


In [ ]:
# [c05] seed_data
# 面向映射正本（規格 §7.1 表 ＋ §7.2 規則代號）：改這裡 → 升 [c02] FACET_MAP_VERSION → 重跑本 notebook（先 dry_run）→ T3 整月重標。
# 格式同 CSV，第一行是欄名、逗號分隔、值不加引號；facet 留空 = 不映射；follow_tag 只給不映射的 tag。
# topic_tag 必須剛好等於 task 3 [c02] TOPIC_TAGS 的 14 個（tests 會比）。
FACET_MAP_SEED = """
topic_tag,facet,relevance_rule,follow_tag,note
demand,market_outlook,always,,§7.1 市場展望；§7.2 規則 1
commercial_relation,market_outlook,always,,§7.1 市場展望（合資 / 商務關係）；規則 1
product_plan,product_tech,always,,§7.1 產品技術；規則 1
spec,product_tech,spec_statement,,§7.1 產品技術；規則 2：statement_type ∈ {plan decision request} 才相關
price,price,always,,§7.1 價格；規則 1
cost,price,always,,§7.1 價格（成本）；規則 1
supply_chain,other,always,,§7.1 其他；規則 1
capacity,other,capacity_timeref,,§7.1 其他；規則 3：time_refs 非空才相關
allocation,other,always,,§7.1 其他；規則 1
quality,other,quality_request_or_customer,,§7.1 其他；規則 4：request 或客戶方發言才相關
schedule,,never,product_plan,§7.1 不映射；與 product_plan 共存則隨 product_plan 映入
engineering,,never,,§7.1 不映射；單獨出現一律不相關
terminology,,never,,§7.1 不映射
admin,,never,,§7.1 不映射
"""


In [ ]:
# [c06] view_specs
# 正本。select = [(expr, 輸出欄名, COMMENT)]；body(fq) 回傳 FROM 起的 SQL，fq(表名) 給完整名稱。
# 四個 View：
#   v_meeting_sb_brief_text  = Sales Brief 文字端一格一列（T4 的填報端輸入；brand = ALL 來自 app_summary）
#   v_meeting_sb_period      = 每 app × 期別的品牌清單與 submitted_at 截止點（T3 算 D6 窗口、brand_in_sb 的材料）
#   v_meeting_items_sb_q     = 標註表（active）⋈ 中立層：補 quote / numbers / model_nos / time_refs（T4 的會議端輸入、PM 看證據）
#   v_meeting_sb_reconcile_q = 對照格表的消費介面：例外清單排序、PM 有效 relation、待審旗標
# 任何 View 不出現姓名 / email（[c03] validate_specs 會擋）。
_SUMMARY_COLS = ", ".join(SB_FACET_COLS.values())
_STACK_ARGS = ",\n    ".join(f"'{f}', b.{c}" for f, c in SB_FACET_COLS.items())
_SB_TEXT = "nullif(trim(f.sb_text_raw), '')"


def _period_expr(year: str, month: str) -> str:
    """fill_year / fill_month → 'yyyy-MM'。"""
    return f"concat(cast({year} AS STRING), '-', lpad(cast({month} AS STRING), 2, '0'))"


def _brief_text_body(fq):
    # 同 app × 品牌 × 期別若有多列（重填），取 create_time 最新的一列。
    return f"""FROM (
  SELECT app_code, brand_code AS brand, fill_year, fill_month, create_time, 'g_sales_brief_brand_summary' AS src,
         {_SUMMARY_COLS},
         row_number() OVER (PARTITION BY app_code, brand_code, fill_year, fill_month ORDER BY create_time DESC) AS rn
  FROM {fq('g_sales_brief_brand_summary')}
  UNION ALL
  SELECT app_code, '{BRAND_ALL}' AS brand, fill_year, fill_month, create_time, 'g_sales_brief_app_summary' AS src,
         {_SUMMARY_COLS},
         row_number() OVER (PARTITION BY app_code, fill_year, fill_month ORDER BY create_time DESC) AS rn
  FROM {fq('g_sales_brief_app_summary')}
) b
LATERAL VIEW stack({len(SB_FACET_COLS)},
    {_STACK_ARGS}) f AS facet, sb_text_raw
WHERE b.rn = 1"""


def _period_body(fq):
    return f"""FROM (
  SELECT app_code, fill_year, fill_month,
         array_sort(collect_set(brand_code)) AS brands, count(DISTINCT brand_code) AS n_brands, max(create_time) AS brand_summary_at
  FROM {fq('g_sales_brief_brand_summary')}
  GROUP BY app_code, fill_year, fill_month
) bs
FULL OUTER JOIN (
  SELECT app_code, fill_year, fill_month,
         min(submitted_at) AS submitted_start, max(submitted_at) AS submitted_end, count(*) AS n_intel_rows
  FROM {fq('g_sales_brief_market_intelligence')}
  GROUP BY app_code, fill_year, fill_month
) mi ON mi.app_code = bs.app_code AND mi.fill_year = bs.fill_year AND mi.fill_month = bs.fill_month
LEFT JOIN (
  SELECT app_code, fill_year, fill_month, max(create_time) AS app_summary_at
  FROM {fq('g_sales_brief_app_summary')}
  GROUP BY app_code, fill_year, fill_month
) a ON a.app_code = coalesce(bs.app_code, mi.app_code)
   AND a.fill_year = coalesce(bs.fill_year, mi.fill_year)
   AND a.fill_month = coalesce(bs.fill_month, mi.fill_month)"""


_P_APP = "coalesce(bs.app_code, mi.app_code)"
_P_YEAR = "coalesce(bs.fill_year, mi.fill_year)"
_P_MONTH = "coalesce(bs.fill_month, mi.fill_month)"


def _items_sb_q_body(fq):
    return f"""FROM {fq('g_meeting_items_sb_bu')} l
LEFT JOIN {fq('g_meeting_items_bu')} i ON i.item_id = l.item_id
WHERE l.active = true"""


def _reconcile_q_body(fq):
    return f"""FROM {fq('g_meeting_sb_reconcile_bu')} r"""


_RANK_EXPR = "CASE r.relation " + " ".join(f"WHEN '{k}' THEN {v}" for k, v in RELATION_RANK.items()) + " ELSE 9 END"

VIEWS = {
    "v_meeting_sb_brief_text": {
        "comment": "Sales Brief 文字端一格一列（BU）：app_name × brand（ALL = app 層，取 g_sales_brief_app_summary；其餘取 g_sales_brief_brand_summary）× facet × brief_month → sb_text。sb_text NULL = 該格空白；品牌有列但四格都 NULL 仍算「該期有此品牌」。W2 T4 的填報端輸入。正本：004 [c06]。",
        "body": _brief_text_body,
        "select": [
            ("b.app_code", "app_name", "TV / NB / MNT（= Sales Brief app_code）"),
            ("b.brand", "brand", "Sales Brief brand_code；ALL = app 層格"),
            ("f.facet", "facet", "market_outlook / product_tech / price / other（對應 summary_futureOutlook / techProduct / pricing / otherInfo）"),
            (_period_expr("b.fill_year", "b.fill_month"), "brief_month", "yyyy-MM（= fill_year-fill_month）"),
            (_SB_TEXT, "sb_text", "該格填報原文；空白 → NULL"),
            (f"CASE WHEN {_SB_TEXT} IS NULL THEN 'none' ELSE b.src END", "sb_source", "來源表名或 none"),
            ("b.create_time", "sb_create_time", "填報表該列的 create_time（同格多列取最新）"),
        ],
    },
    "v_meeting_sb_period": {
        "comment": "Sales Brief 每 app × 期別的材料（BU）：品牌清單（brand_in_sb 用）、market_intelligence 的 submitted_at 起迄（D6：以 submitted_end 當該期截止點，prev_submitted_end 當上一期截止點）、是否有 app 層摘要。W2 T3 算窗口用。正本：004 [c06]。",
        "body": _period_body,
        "select": [
            (_P_APP, "app_name", "TV / NB / MNT"),
            (_period_expr(_P_YEAR, _P_MONTH), "brief_month", "yyyy-MM"),
            ("coalesce(bs.brands, cast(array() AS ARRAY<STRING>))", "brands", "該期 brand_summary 出現的 brand_code（排序）；無 → 空陣列"),
            ("coalesce(bs.n_brands, 0)", "n_brands", "品牌數"),
            ("mi.submitted_start", "submitted_start", "該期 market_intelligence 最早送出時間"),
            ("mi.submitted_end", "submitted_end", "該期 market_intelligence 最晚送出時間 = D6 截止點；NULL = 該期沒有填報"),
            (f"lag(mi.submitted_end) OVER (PARTITION BY {_P_APP} ORDER BY {_P_YEAR}, {_P_MONTH})", "prev_submitted_end",
             "上一期的截止點（窗口起 = 它 + 1 日）；第一期 NULL"),
            ("mi.n_intel_rows", "n_intel_rows", "該期 market_intelligence 列數（品牌 × 四面向）"),
            ("a.app_summary_at IS NOT NULL", "has_app_summary", "該期有 app 層摘要（ALL 格有填報端）"),
            ("bs.brand_summary_at", "brand_summary_at", "brand_summary 該期最新 create_time"),
            ("a.app_summary_at", "app_summary_at", "app_summary 該期最新 create_time"),
        ],
    },
    "v_meeting_items_sb_q": {
        "comment": "Sales Brief 標註列（BU，只留 active）⋈ 中立層：補 quote / numbers / model_nos / time_refs / customer_raw。W2 T4 的會議端輸入、PM 看證據、Genie 可掛。不含姓名 email。正本：004 [c06]。",
        "body": _items_sb_q_body,
        "select": [(f"l.{n}", n, c) for n, _, c in TABLES["g_meeting_items_sb_bu"]["columns"]] + [
            ("i.quote", "quote", "逐字原文（已 token 化）"),
            ("i.numbers", "numbers", "JSON 字串 [{value, unit, what}]；shareable 驗證用"),
            ("i.time_refs", "time_refs", "JSON 字串 [{when, what}]；§7.2 規則 3 用"),
            ("i.model_nos", "model_nos", "提及的型號"),
            ("i.customer_raw", "customer_raw", "素材原名"),
            ("i.subject_clean", "subject_clean", "會議名稱"),
            ("i.app", "app_neutral", "中立層原始 app（TV / NB / MNT / ALL / NULL）"),
            ("i.extract_status = 'ok' AND i.superseded = false", "item_still_valid", "中立層此 item_id 仍有效；false = 已被重萃取代，等 T3 重標"),
        ],
    },
    "v_meeting_sb_reconcile_q": {
        "comment": "Sales Brief 對照格消費介面（BU）：加 §4.6 例外清單順序 relation_rank（conflict 1 → supplement 2 → meeting_only 3 → corroborate 4 → sb_only 5）、PM 修正後的 effective_relation、待審旗標。meeting_summary 含 S2，對外只用 meeting_summary_shareable。正本：004 [c06]。",
        "body": _reconcile_q_body,
        "select": [(f"r.{n}", n, c) for n, _, c in TABLES["g_meeting_sb_reconcile_bu"]["columns"]] + [
            (_RANK_EXPR, "relation_rank", "§4.6 清單順序：conflict 1 / supplement 2 / meeting_only 3 / corroborate 4 / sb_only 5"),
            (f"r.brand = '{BRAND_ALL}'", "is_app_level", "ALL 格（跨客戶訊號）"),
            ("coalesce(r.review_relation, r.relation)", "effective_relation", "PM fix_relation 後的有效 relation；未審 = relation"),
            ("r.review_verdict IS NULL AND r.relation <> 'sb_only'", "review_pending", "待 PM 審（sb_only 不審）"),
        ],
    },
}


In [ ]:
# [c10] build_ddl
# 驗 spec 與種子 → 組 DDL → 看前置表與既有物件；dry_run 只印不建。
from pyspark.sql import functions as F

errors = validate_specs(TABLES, VIEWS)
facet_rows = parse_csv_text(FACET_MAP_SEED, FACET_MAP_COLS)
errors += validate_facet_seed(facet_rows)
for e in errors:
    print("ERROR", e)
assert not errors, f"spec / 種子驗證失敗 {len(errors)} 筆，修 [c04] / [c05] / [c06] 後重跑"


def fq(name: str) -> str:
    return f"{catalog}.{schema}.{name}"


# View 引用的前置表（002 的 items 表、Sales Brief 三張表）；T3 另外還要 001 的 s_semantic_entity / _alias。
PREREQ_TABLES = ("g_meeting_items_bu", "g_sales_brief_brand_summary", "g_sales_brief_app_summary",
                 "g_sales_brief_market_intelligence")
missing_prereq = [t for t in PREREQ_TABLES if not spark.catalog.tableExists(fq(t))]
print("前置表缺少：", missing_prereq or "無")

table_ddl = {t: build_create_table(fq(t), s) for t, s in TABLES.items()}
view_ddl = {v: build_create_view(fq(v), s, fq) for v, s in VIEWS.items()}
existing = {t: spark.catalog.tableExists(fq(t)) for t in list(TABLES) + list(VIEWS)}
print("既有物件：", {k: v for k, v in existing.items() if v} or "無")
print("將建立：", [k for k, v in existing.items() if not v])

# 種子差異（表已存在時）：新增 / 內容變更 / 將停用
seed_schema = ", ".join(f"{c} string" for c in FACET_MAP_COLS)
df_seed = spark.createDataFrame([tuple(r[c] for c in FACET_MAP_COLS) for r in facet_rows], seed_schema)
if existing["s_meeting_sb_facet_map"]:
    old = spark.table(fq("s_meeting_sb_facet_map"))
    j = df_seed.alias("n").join(old.alias("o"), "topic_tag", "full_outer")
    same = F.lit(True)
    for c in FACET_MAP_COLS[1:]:
        same = same & F.col(f"n.{c}").eqNullSafe(F.col(f"o.{c}"))
    flags = {
        "insert": F.col("o.is_active").isNull() & F.col("n.relevance_rule").isNotNull(),
        "update": F.col("n.relevance_rule").isNotNull() & F.col("o.is_active").isNotNull() & (~same | ~F.col("o.is_active")),
        "deactivate": F.col("n.relevance_rule").isNull() & F.col("o.is_active"),
    }
    r = j.agg(*[F.sum(x.cast("int")).alias(k) for k, x in flags.items()]).collect()[0]
    seed_diff = {k: int(r[k] or 0) for k in flags}
else:
    seed_diff = {"insert": len(facet_rows), "update": 0, "deactivate": 0}
print(f"種子 {len(facet_rows)} 列（map {FACET_MAP_VERSION}），預計變更：", seed_diff)

if dry_run:
    print("dry_run = true：只印 DDL 與種子，不建任何物件。確認後把 dry_run 改 false 重跑。\n")
    for ddl in list(table_ddl.values()) + list(view_ddl.values()):
        print(ddl, end=";\n\n")
    df_seed.show(len(facet_rows), truncate=False)
else:
    assert not missing_prereq, f"前置表缺少 {missing_prereq}：先跑 002 / 確認 Sales Brief 表名與 schema"


In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。既有表的欄位差異由 [c30] 報告，不在這裡 ALTER。
if not dry_run:
    for t, ddl in table_ddl.items():
        spark.sql(ddl)
        print(("exists " if existing[t] else "created") + " " + fq(t))


In [ ]:
# [c21] merge_seed
# 面向映射種子 MERGE：鍵相同且內容有變（或曾被停用）才更新 updated_at；種子拿掉的列標 is_active = false（不刪，保留回溯）。
if not dry_run:
    df_seed.createOrReplaceTempView("_seed_sb_facet_map")
    cmp = FACET_MAP_COLS[1:]
    differs = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cmp) + " OR NOT t.is_active"
    spark.sql(f"""
        MERGE INTO {fq('s_meeting_sb_facet_map')} t USING _seed_sb_facet_map s ON t.topic_tag = s.topic_tag
        WHEN MATCHED AND ({differs}) THEN UPDATE SET
            {", ".join(f"t.{c} = s.{c}" for c in cmp)}, t.is_active = true, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT ({", ".join(FACET_MAP_COLS)}, is_active, updated_at)
            VALUES ({", ".join(f"s.{c}" for c in FACET_MAP_COLS)}, true, current_timestamp())
        WHEN NOT MATCHED BY SOURCE AND t.is_active THEN UPDATE SET
            t.is_active = false, t.updated_at = current_timestamp()
    """)
    print("seed merged", seed_diff)


In [ ]:
# [c22] create_views
# View 無狀態：CREATE OR REPLACE，改 [c06] 後重跑即生效。必須在 [c20] 之後（View 引用本支兩張表與前置表）。
if not dry_run:
    for v, ddl in view_ddl.items():
        spark.sql(ddl)
        print(("replaced" if existing[v] else "created ") + " " + fq(v))


In [ ]:
# [c23] write_log
# 每次正式執行（dry_run = false）寫一列 b_meeting_migration_log；只 INSERT，不改舊列。
if not dry_run:
    result = {
        "tables": {t: ("exists" if existing[t] else "created") for t in TABLES},
        "views": {v: ("replaced" if existing[v] else "created") for v in VIEWS},
        "seed": {"facet_map_version": FACET_MAP_VERSION, **seed_diff},
    }
    spark.sql(
        f"INSERT INTO {fq('b_meeting_migration_log')} "
        "SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result, ensure_ascii=False), "note": None},
    )
    print("logged", result)


In [ ]:
# [c30] check_target
# 比對實際表欄位 vs spec：缺欄（表早已存在、spec 後來加欄）→ 印補欄語句並 fail；型別不同 → 警告。
# View：能查、輸出欄 = spec、不含 S3 欄。種子：active 列數 = [c05] 列數。最後印每 app 最近三期的 Sales Brief 材料，確認窗口與品牌清單合理。
if not dry_run:
    problems = []
    for t, spec in TABLES.items():
        actual = {f.name: f.dataType.simpleString() for f in spark.table(fq(t)).schema.fields}
        missing = [n for n, _, _ in spec["columns"] if n not in actual]
        if missing:
            problems.append(f"{t} 缺欄 {missing}。補欄（開新流水號執行）：\n  {add_columns_sql(fq(t), spec, missing)}")
        for n, typ, _ in spec["columns"]:
            if n in actual and actual[n] != typ.lower():
                print(f"WARN {t}.{n} 型別 spec={typ.lower()} actual={actual[n]}")
        extra = [n for n in actual if n not in {c[0] for c in spec["columns"]}]
        if extra:
            print(f"INFO {t} 有 spec 以外的欄（不處理）：{extra}")
    for v, spec in VIEWS.items():
        cols = spark.table(fq(v)).columns
        want = [a for _, a, _ in spec["select"]]
        if cols != want:
            problems.append(f"{v} 輸出欄與 spec 不同：{cols}")
        leak = sorted(S3_COLS & set(cols))
        if leak:
            problems.append(f"{v} 暴露 S3 欄：{leak}")
        spark.table(fq(v)).limit(1).collect()  # 能執行（表空也要能跑）
    n_active = spark.table(fq("s_meeting_sb_facet_map")).where("is_active").count()
    if n_active != len(facet_rows):
        problems.append(f"s_meeting_sb_facet_map active {n_active} ≠ 種子 {len(facet_rows)}")
    for p in problems:
        print("PROBLEM", p)
    assert not problems, f"{len(problems)} 個問題"
    print("tables:", {t: len(spark.table(fq(t)).columns) for t in TABLES})
    print("views :", {v: len(spark.table(fq(v)).columns) for v in VIEWS})
    print("facet_map active:", n_active)
    # 每 app 最近三期：品牌清單、截止點、上一期截止點（D6 窗口的材料；表小，直接看）
    from pyspark.sql.window import Window

    (spark.table(fq("v_meeting_sb_period"))
     .withColumn("rn", F.row_number().over(Window.partitionBy("app_name").orderBy(F.desc("brief_month"))))
     .where("rn <= 3").drop("rn")
     .select("app_name", "brief_month", "n_brands", "brands", "submitted_end", "prev_submitted_end", "has_app_summary")
     .orderBy("app_name", "brief_month")
     .show(12, truncate=False))
